##### Consider the following input sentence, which has already been embedded into three-dimensional vectors

In [3]:
import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your     (x^1)
     [0.55, 0.87, 0.66], # journey  (x^2)
     [0.57, 0.85, 0.64], # starts   (x^3)
     [0.22, 0.58, 0.33], # with     (x^4)
     [0.77, 0.25, 0.10], # one      (x^5)
     [0.05, 0.80, 0.55]  # step     (x^6)
    ]
)

The first step of implementing `self-ttention` is to compute the intermdiate values `w`, referred to as `attention scores`.

In [4]:
query = inputs[1] # The second input token serves as the query
attn_scores_2 = torch.empty(inputs.shape[0]) # Initialize an empty tensor to store attention scores

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, x_i) # Compute the dot product between the query and each input token
    
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Understanding `dot products`

A `dot product` is essentialy a concise way of multiplying two vectors element-wise and then summing the products, which can be demonstrated as follows :

In [6]:
res = 0
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)

tensor(0.9544)


In [8]:
print(torch.dot(inputs[0], query))

tensor(0.9544)


The output confirms that sum of the element-wise multiplication gives the same results as the dot product.

---
Dans le contexte des mécanismes d’auto-attention, le `produit scalaire (dot product)` permet de déterminer dans quelle mesure un élément d’une séquence "porte attention" à un autre.
Plus le produit scalaire est grand, plus la similarité (et donc l’attention) entre deux éléments est importante.

---
In the next step, we `normalize` each of the **attention scores** we compute previously. The `main goal` of the normalization is `to obtain` **attention weights** `that sum up to 1`. This normalization is a convention that `is useful for interpretation and maintaining training stability` in an LLM.

In [9]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In practice, `it's more common and advisable to use` **softmax function** `for normalization`. This approach `is better at managing extreme values and offers more favorable gradient properties` during training.

In [10]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention_weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention_weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In addition, the `softmax function ensures that the attention weights are always positive`. This makes the output **interpretable as probabilities or relative importance**, where **higher weights indicate greater importance**.


Note that this naive softmax implementation (`softmax_naive`) may `encounter numerical instability problems` such as **overflow** and **underflow**, when dealing with large or small input value.

Therefore in practice, it's advisable to `use the Pytorch implementation of sofmax`, which has been extensively optimized for performance.

In [12]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention_weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention_weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


Npw that we have computed the normalized attention weights, we are going to calculate the **context vector** $z^{(2)}$ by **multiplying the embeddeed input token $x(i)$**, with **the corresponding attention weights and then summing the vectors**.

In [ ]:
query = inputs[1] # The second input token is the query
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])
